# Getting started with LLM4S in Jupyter

A completion, a streamed answer, structured output and a typed tool, in the Almond Scala kernel.
The notebook runs **offline** as committed: every answer comes from `llm4s-testkit`'s `ScriptedLLMClient`, so it
needs no API key and no model server. The cell marked **Switch to a real provider** points it at a real model.

Setup (Almond, the kernel, JDK 21): [Notebooks guide](https://llm4s.org/guide/notebooks).

In [ ]:
//> using dep org.llm4s::llm4s-core:0.5.1
//> using dep org.llm4s::llm4s-schema-derivation:0.5.1
//> using dep org.llm4s::llm4s-testkit:0.5.1

## The scripted client

`ScriptedLLMClient.sequence` answers the first call with the first reply, the second with the second, and so on.

In [ ]:
import org.llm4s.llmconnect.LLMClient
import org.llm4s.llmconnect.model.*
import org.llm4s.testkit.{ Reply, ScriptedLLMClient }

val scripted = ScriptedLLMClient
  .sequence(
    Reply.text("Covariance lets a List[Cat] be used where a List[Animal] is expected."),
    Reply.text("Scala 3 brings enums, opaque types, extension methods, given instances and a new indentation syntax."),
    Reply.text("""{"vendor":"ACME","amount":12.5,"currency":"EUR"}"""),
    Reply.toolCall("get_weather", """{"city":"Paris"}"""),
    Reply.text("It is 21 degrees in Paris.")
  )
  .withChunkSize(12)

## Switch to a real provider

Set `useRealProvider = true` to call the provider that `llm4s.providers.provider` names. Configuration comes from system
properties set in this cell before the first load: `config.file` naming an `application.conf`, or single settings; add
the provider's module to the first cell (for Ollama, `//> using dep org.llm4s::llm4s-ollama:<version>`) and restart
the kernel. The `assert`s below check only the scripted answers, so they are skipped for a real model.

In [ ]:
// SWITCH TO A REAL PROVIDER: set to true
val useRealProvider = false

// An application.conf:
// System.setProperty("config.file", "/path/to/application.conf")
// Or, for a local Ollama server, single settings:
// System.setProperty("llm4s.providers.provider", "ollama-local")
// System.setProperty("llm4s.providers.ollama-local.provider", "ollama")
// System.setProperty("llm4s.providers.ollama-local.model", "llama3.2")

val client: LLMClient =
  if (!useRealProvider) scripted
  else {
    import org.llm4s.config.Llm4sConfig
    import org.llm4s.llmconnect.LLMConnect
    import org.llm4s.model.ModelRegistryService
    (for {
      registry <- Llm4sConfig.modelRegistryService()
      given ModelRegistryService = registry
      config <- Llm4sConfig.defaultProvider()
      real   <- LLMConnect.getClient(config)
    } yield real).fold(error => sys.error(error.formatted), identity)
  }

## A completion

In [ ]:
val answer = for {
  conversation <- Conversation.userOnly("Explain variance in Scala 3 in one sentence")
  completion   <- client.complete(conversation)
} yield completion.content

answer.fold(error => println(error.formatted), println)
if (!useRealProvider) assert(answer == Right("Covariance lets a List[Cat] be used where a List[Animal] is expected."))

## Streaming into the cell

`streamComplete` calls back with each chunk; one `Markdown` display is updated in place as the text grows.

In [ ]:
import almond.display.Markdown

val shown    = Markdown("")
shown.display()
var streamed = ""

val streamResult = for {
  conversation <- Conversation.userOnly("What does Scala 3 add? One sentence.")
  completion <- client.streamComplete(
    conversation,
    CompletionOptions(),
    chunk => chunk.content.foreach { text =>
      streamed += text
      shown.withContent(streamed).update()
    }
  )
} yield completion.content

if (!useRealProvider) assert(streamResult == Right(streamed) && streamed.startsWith("Scala 3 brings enums"))

## Structured output

`derives SchemaOf` gives the JSON schema sent to the model; `completeStructuredOf` reads the reply back as the type.

In [ ]:
import org.llm4s.schema.*
import upickle.default.ReadWriter

case class Invoice(vendor: String, amount: Double, currency: String) derives SchemaOf, ReadWriter

val invoice = for {
  conversation <- Conversation.userOnly("Extract the invoice: ACME, 12.50 EUR, two widgets")
  invoice      <- client.completeStructuredOf[Invoice](conversation)
} yield invoice

invoice.fold(error => println(error.formatted), println)
if (!useRealProvider) assert(invoice == Right(Invoice("ACME", 12.5, "EUR")))

## A typed tool

The tool's arguments are a case class: its schema is derived, and the handler reads the call's JSON back into it.
The cell runs the tool the model asks for, then sends the result back for the final answer.

In [ ]:
import org.llm4s.toolapi.{ ToolBuilder, ToolCallRequest, ToolRegistry }

case class WeatherQuery(city: String) derives SchemaOf, ReadWriter
case class Weather(city: String, celsius: Int) derives ReadWriter

val weatherTool = ToolBuilder[WeatherQuery, Weather]("get_weather", "The weather in a city", SchemaOf[WeatherQuery].definition)
  .withHandler(args =>
    scala.util.Try(upickle.default.read[WeatherQuery](args.params)).toEither.left
      .map(_.getMessage)
      .map(query => Weather(query.city, 21))
  )
  .buildSafe()

In [ ]:
val toolAnswer = for {
  tool         <- weatherTool
  tools         = new ToolRegistry(Seq(tool))
  options       = CompletionOptions().withTools(tools.tools)
  conversation <- Conversation.userOnly("What is the weather in Paris?")
  first        <- client.complete(conversation, options)
  results = first.toolCalls.map { call =>
    val result = tools.execute(ToolCallRequest(call.name, call.arguments))
    println(s"-> ${call.name}(${call.arguments}) <- ${result.fold(_.toString, _.render())}")
    ToolMessage(result.fold(_.toString, _.render()), call.id)
  }
  last <- client.complete(conversation.addMessage(first.message).addMessages(results), options)
} yield last.content

toolAnswer.fold(error => println(error.formatted), println)
if (!useRealProvider) assert(toolAnswer == Right("It is 21 degrees in Paris.") && scripted.callCount == 5)